# Segmentation d'images avec Keras

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy
import pandas
import tensorflow as tf
from tensorflow import keras
from tifffile import imread

In [ ]:
!nvidia-smi

In [ ]:
%%time
!rm -rf sample_data
!git clone https://github.com/mlambda/ign-FNF-2009.git

## Chargement des données

Les données sont des photographies aériennes prises par l'IGN en 2009 en Lorraine : 20 zones de 5 km × 5 km. Chaque zone est découpée en 9 × 9 = 81 tuiles de 1024 × 1024 pixels (1 pixel = 50 cm sur le terrain : une tuile couvre 512 m × 512 m).

- **Entrée** (fichiers `*_irc.tif`) : une image en *infrarouge couleur* (IRC). Ses trois canaux sont le proche infrarouge, le rouge et le vert, affichés en rouge, vert et bleu. La végétation réfléchit fortement l'infrarouge : elle apparaît en rouge.
- **Cible** (fichiers `*_pfnf_nch.tif`) : la classe de chaque pixel, forêt ou non-forêt, codée en *one-hot* sur deux canaux. Le code en garde l'indice : 1 pour forêt, 0 pour non-forêt. Au sens de l'IGN, une forêt n'est pas un simple groupe d'arbres : c'est une surface de plus de 5 000 m², large d'au moins 20 m, dont la végétation couvre au moins 10 % du sol et qui n'est pas utilisée pour l'agriculture.

2 zones ont été mises de côté pour la validation, et 2 autres pour le test. Toutes les tuiles d'une zone vont dans le même ensemble : des tuiles voisines se ressemblent trop pour mesurer honnêtement la généralisation.

In [ ]:
df_train = pandas.read_csv("ign-FNF-2009/train-files.csv")
df_val = pandas.read_csv("ign-FNF-2009/val-files.csv")
df_test = pandas.read_csv("ign-FNF-2009/test-files.csv")

print("Coordonnées en entraînement")
print(df_train.coord.unique())

print("Coordonnées en val")
print(df_val.coord.unique())

print("Coordonnées en test")
print(df_test.coord.unique())

df_test

## Étude d'un exemple et de son annotation

In [ ]:
example_input = imread(df_train.Observations[0])
example_annotation = imread(df_train.Annotations[0])

In [ ]:
def display_sample(*images: numpy.array) -> None:
  """Show side-by-side an input image, the ground truth and the prediction."""
  figure, axes = plt.subplots(1, len(images), figsize=(18, 18))

  titles = ["Entrée", "Cible", "Prédiction"]
  for image, ax, title in zip(images, axes, titles):
    ax.set_title(title)
    if image.ndim == 3 and image.shape[2] == 3:
      # Image en infrarouge couleur
      ax.imshow(image)
    else:
      if image.ndim == 3:
        # Canal de la classe forêt (le dernier)
        image = image[..., -1]
      # Bornes fixes : forêt (1) en jaune, non-forêt (0) en violet, même quand
      # toute la tuile est d'une seule classe
      ax.imshow(image, vmin=0, vmax=1)
    ax.axis("off")
  plt.show()


display_sample(example_input, example_annotation)

## Options générales

In [ ]:
# Dimension des images originales
dim_original = 1024

# Dimension des images en entrée du réseau
dim_input = 512

# Nombre de canaux des images en entrée
num_channel = 3

## Augmentation des données

En traitement d'images, il est extrêmement important d'augmenter les données par des transformations pour rendre le réseau appris moins sensibles aux rotations, recadrages, etc. Implémentez la fonction `transform_data`, qui, si `eval` vaut `True` se contentera de faire un recadrage centré des images `X` et `Y`, et sinon fera toutes les transformations que vous jugerez opportunes sur ces mêmes tableaux pour l'entraînement (recadrage aléatoire, miroir, rotation, bruitage des couleurs, de la luminosité, etc).

Pour ce faire, utilisez les fonctions du module [`tf.image`](https://www.tensorflow.org/api_docs/python/tf/image/).

In [ ]:
def transform_data(
  X: numpy.ndarray,
  Y: numpy.ndarray,
  eval: bool = False,
  cropped_size: tuple[int, int] = (dim_input, dim_input),
) -> tuple[numpy.ndarray, numpy.ndarray]:
  # Votre code ici
  return X, Y


# display_sample(*transform_data(example_input, example_annotation))

### Solution

In [ ]:
def transform_data(
  X: numpy.ndarray,
  Y: numpy.ndarray,
  eval: bool = False,
  cropped_size: tuple[int, int] = (dim_input, dim_input),
) -> tuple[numpy.ndarray, numpy.ndarray]:
  height, width = X.shape[:2]
  cropped_height, cropped_width = cropped_size
  if eval:
    # Recadrage centré
    crop_params = dict(
      offset_height=(height - cropped_height) // 2,
      offset_width=(width - cropped_width) // 2,
      target_height=cropped_height,
      target_width=cropped_width,
    )
    X = tf.image.crop_to_bounding_box(X, **crop_params)
    Y = tf.image.crop_to_bounding_box(Y, **crop_params)
  else:
    # Rotation aléatoire
    k = random.choice([None, 1, 3])
    if k is not None:
      X = tf.image.rot90(X, k)
      Y = tf.image.rot90(Y, k)

    # Recadrage aléatoire
    crop_params = dict(
      offset_height=random.randint(0, height - cropped_height),
      offset_width=random.randint(0, width - cropped_width),
      target_height=cropped_height,
      target_width=cropped_width,
    )
    X = tf.image.crop_to_bounding_box(X, **crop_params)
    Y = tf.image.crop_to_bounding_box(Y, **crop_params)

    # Miroir horizontal
    if random.random() > 0.5:
      X = tf.image.flip_left_right(X)
      Y = tf.image.flip_left_right(Y)

    # Miroir vertical
    if random.random() > 0.5:
      X = tf.image.flip_up_down(X)
      Y = tf.image.flip_up_down(Y)
  return X, Y


for _ in range(10):
  display_sample(*transform_data(example_input, example_annotation))

## Création d'un générateur Keras

Pour définir un générateur Keras, nous allons utiliser la classe [`keras.utils.Sequence`](https://www.tensorflow.org/api_docs/python/tf/keras/utils/Sequence). Pour ce faire, Il faut principalement deux méthodes : `__getitem__`, qui est appelée à chaque itération pour récupérer le batch à traiter, et `__len__`, qui est appelée pour connaître le nombre total de batchs qui sera fourni par le générateur. Nous utiliserons aussi `on_epoch_end` pour mélanger le dataset avant l'itération suivante.

In [ ]:
class DataGenerator(keras.utils.Sequence):
  """Generate data in batches with appropriate transforms."""

  def __init__(
    self,
    df: pandas.DataFrame,
    batch_size: int,
    eval: bool,
    cropped_size: tuple[int, int] = (dim_input, dim_input),
  ) -> None:
    """Initialize the generator."""
    # Votre code ici

  def __len__(self):
    """Compute the number of batches per epoch."""
    # Votre code ici

  def __getitem__(self, index):
    """Generate one batch of data."""
    # Votre code ici

  def on_epoch_end(self):
    """Update indexes after each epoch."""
    # Votre code ici

### Solution

In [ ]:
class DataGenerator(keras.utils.Sequence):
  """Generate data for Keras."""

  def __init__(
    self,
    df: pandas.DataFrame,
    batch_size: int,
    eval: bool,
    cropped_size: tuple[int, int] = (dim_input, dim_input),
  ) -> None:
    """Initialize the generator."""
    self.X_paths = df.Observations
    self.Y_paths = df.Annotations
    assert self.X_paths.shape[0] == self.Y_paths.shape[0]
    self.batch_size = batch_size
    self.eval = eval
    self.cropped_size = cropped_size
    if eval:
      self.indices = numpy.arange(self.X_paths.shape[0])
    else:
      self.indices = numpy.random.permutation(self.X_paths.shape[0])

  def __len__(self):
    """Compute the number of batches per epoch."""
    return int(math.ceil(self.X_paths.shape[0] / self.batch_size))

  def __getitem__(self, index):
    """Generate one batch of data."""
    # Generate indexes of the batch
    start = index * self.batch_size
    Xs = []
    Ys = []
    for i in self.indices[start : start + self.batch_size]:
      X, Y = transform_data(
        imread(self.X_paths[i]),
        imread(self.Y_paths[i]).argmax(axis=-1)[..., numpy.newaxis],
        self.eval,
        self.cropped_size,
      )
      Xs.append(X)
      Ys.append(tf.squeeze(Y))
    return tf.stack(Xs), tf.stack(Ys)

  def on_epoch_end(self):
    """Update indexes after each epoch."""
    if not self.eval:
      self.indices = numpy.arange(self.X_paths.shape[0])


generator = DataGenerator(df_train, 3, True)
X, Y = generator[0]
X.shape, Y.shape

## Création du modèle

Pour le modèle, nous devons passer d'une image de dimension (batch × hauteur × largeur × 3) à une sortie de dimension (batch × hauteur × largeur).

Vous pouvez vous inspirer de l'architecture vue en démonstration des autoencodeurs débruiteurs. Cette fois cependant, on ne réduira pas les capacités du réseau au centre du modèle.

Vous pouvez aussi consulter [cette page d'exemple de code Keras](https://keras.io/examples/vision/oxford_pets_image_segmentation/#prepare-unet-xceptionstyle-model) pour plus d'insipration.

In [ ]:
def get_model(starting_depth: int = 32, steps: int = 5) -> keras.Model:
  pass  # Votre code ici

### Solution

In [ ]:
# Implémentation du modèle U-Net décrit dans l'article
# https://arxiv.org/abs/1505.04597
# Vous pouvez aussi implémenter tout autre modèle adapté, voir par exemple ce
# panorama : https://arxiv.org/abs/2001.05566


def downsample(inputs):
  return keras.layers.MaxPooling2D(pool_size=(2, 2))(inputs)


def upsample(inputs):
  return keras.layers.UpSampling2D(size=(2, 2))(inputs)


def semiblock(inputs, feature_maps):
  conv = keras.layers.Conv2D(feature_maps, 3, activation="relu", padding="same")(inputs)
  batch_norm = keras.layers.BatchNormalization()(conv)
  relu = keras.layers.ReLU()(batch_norm)
  dropout = keras.layers.Dropout(0.1)(relu)
  return dropout


def block(inputs, feature_maps):
  return semiblock(semiblock(inputs, feature_maps), feature_maps)


def concat(*layers):
  return keras.layers.Concatenate()(layers)


def get_model(starting_depth: int = 32, steps: int = 5) -> keras.Model:
  inputs = keras.layers.Input((dim_input, dim_input, num_channel))
  encodings = []
  depths = [min(512, starting_depth * 2**i) for i in range(steps)]
  for depth in depths:
    encodings.append(block(downsample(encodings[-1]) if encodings else inputs, depth))
  x = encodings[-1]
  for depth, encoding in zip(depths[-2::-1], encodings[-2::-1]):
    x = block(concat(encoding, upsample(x)), depth)
  outputs = keras.layers.Conv2D(1, 1, activation="sigmoid", padding="same")(x)

  model = keras.Model(inputs=inputs, outputs=outputs)

  return model

## Entraînement

On peut maintenant utiliser notre classe de génération de données pour entraîner notre modèle.

En plus de l'accuracy, la part des pixels bien classés, on suit l'IoU (*intersection over union*) de la classe forêt : la surface où la prédiction et la cible disent toutes deux « forêt », divisée par la surface où au moins l'une des deux le dit. Elle vaut 1 pour un masque parfait, et 0 pour un modèle qui ne prédit jamais la forêt, quelle que soit son accuracy.

In [ ]:
# Votre code ici

### Solution

In [ ]:
# Construction du modèle
model = get_model()
model.summary()
model.compile(
  optimizer=keras.optimizers.Adam(learning_rate=1e-4),
  loss="binary_crossentropy",
  metrics=[
    "accuracy",
    keras.metrics.BinaryIoU(target_class_ids=[1], threshold=0.5, name="iou"),
  ],
)

batch_size = 4

# Création des générateurs
train_generator = DataGenerator(df_train, batch_size, eval=False)
val_generator = DataGenerator(df_val, batch_size, eval=True)

# Entraînement du modèle, en évaluation sur le générateur de validation à la fin
# de chaque epoch
epochs = 15
model.fit(
  train_generator,
  validation_data=val_generator,
  epochs=epochs,
  callbacks=[
    keras.callbacks.EarlyStopping(patience=5, monitor="val_loss"),
    keras.callbacks.ModelCheckpoint(
      filepath="best_model.weights.h5",
      monitor="val_accuracy",
      mode="max",
      save_weights_only=True,
      save_best_only=True,
    ),
  ],
)

## Application aux données de test

Une fois votre modèle suffisamment appris, appliquez-le aux données de test : les prédictions sont-elles de bonne qualité ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
# Chargement du meilleur modèle
model.load_weights("best_model.weights.h5")

test_generator = DataGenerator(df_test, 24, eval=True)
X, Y = test_generator[2]
model.evaluate(test_generator, batch_size=4)
preds = model.predict(X)[..., 0]

print(X.shape, Y.shape, preds.shape)

for x, y, p in zip(X, Y, preds):
  display_sample(x, y, p)

## U-Net pré-entraîné

Le modèle précédent apprend tout de zéro, encodeur compris. Un encodeur déjà entraîné sur un autre problème sait extraire des bords, des textures et des formes : c'est l'apprentissage par transfert. [MobileNetV2](https://arxiv.org/abs/1801.04381), entraîné à classer les photos d'ImageNet, servira d'encodeur à un U-Net. On le gèle (`trainable = False`) : seul le décodeur apprend. ImageNet ne contient pas d'images infrarouges, mais ces caractéristiques visuelles restent utiles.

La fonction `get_pretrained_unet` prépare l'encodeur : pour une image de 512 × 512, il rend dans `skips` les caractéristiques calculées à 256, 128, 64 et 32 pixels de côté, et dans `x` le fond du « U », à 16 × 16. Complétez le décodeur : à chaque étape, doublez la résolution (`Conv2DTranspose` avec `strides=2`), concaténez l'élément de `skips` de même résolution (connexion transversale), puis appliquez une convolution. Terminez par un dernier doublement, jusqu'à 512 × 512, et une couche qui donne à chaque pixel une probabilité d'être de la forêt.

In [ ]:
def get_pretrained_unet(
  input_height: int = dim_input, input_width: int = dim_input
) -> keras.Model:
  # Encodeur : MobileNetV2 pré-entraîné sur ImageNet, sans sa tête de
  # classification. Keras prévient qu'il charge les poids prévus pour des images
  # de 224 × 224 : ses convolutions s'appliquent à toute taille d'image.
  mobilenet = keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(input_height, input_width, num_channel),
  )
  # Sorties de l'encodeur reprises par le décodeur, de la plus fine (256 × 256)
  # à la plus grossière (32 × 32), puis le fond du « U » (16 × 16)
  layer_names = [
    "block_1_expand_relu",
    "block_3_expand_relu",
    "block_6_expand_relu",
    "block_13_expand_relu",
    "block_16_project",
  ]
  encoder = keras.Model(
    mobilenet.input,
    [mobilenet.get_layer(name).output for name in layer_names],
    name="encoder",
  )
  encoder.trainable = False

  inputs = keras.Input(shape=(input_height, input_width, num_channel))
  # MobileNetV2 attend des pixels entre -1 et 1
  x = keras.layers.Rescaling(1 / 127.5, offset=-1)(inputs)
  *skips, x = encoder(x)

  # Votre code ici : le décodeur, de x (16 × 16) jusqu'à une sortie de
  # 512 × 512 × 1, en concaténant chaque élément de skips à sa résolution
  outputs = x
  return keras.Model(inputs, outputs, name="unet")

### Solution

In [ ]:
def get_pretrained_unet(
  input_height: int = dim_input, input_width: int = dim_input
) -> keras.Model:
  # Encodeur : MobileNetV2 pré-entraîné sur ImageNet, sans sa tête de
  # classification. Keras prévient qu'il charge les poids prévus pour des images
  # de 224 × 224 : ses convolutions s'appliquent à toute taille d'image.
  mobilenet = keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(input_height, input_width, num_channel),
  )
  # Sorties de l'encodeur reprises par le décodeur, de la plus fine (256 × 256)
  # à la plus grossière (32 × 32), puis le fond du « U » (16 × 16)
  layer_names = [
    "block_1_expand_relu",
    "block_3_expand_relu",
    "block_6_expand_relu",
    "block_13_expand_relu",
    "block_16_project",
  ]
  encoder = keras.Model(
    mobilenet.input,
    [mobilenet.get_layer(name).output for name in layer_names],
    name="encoder",
  )
  encoder.trainable = False

  inputs = keras.Input(shape=(input_height, input_width, num_channel))
  # MobileNetV2 attend des pixels entre -1 et 1
  x = keras.layers.Rescaling(1 / 127.5, offset=-1)(inputs)
  *skips, x = encoder(x)

  # Décodeur : à chaque étape, on double la résolution et on concatène la sortie
  # de l'encodeur de même résolution (connexion transversale)
  for filters, skip in zip([256, 128, 64, 32], reversed(skips)):
    x = keras.layers.Conv2DTranspose(filters, 3, strides=2, padding="same")(x)
    x = keras.layers.Concatenate()([x, skip])
    x = keras.layers.Conv2D(filters, 3, padding="same")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation("relu")(x)

  # Dernier doublement, jusqu'à la résolution de l'image d'entrée
  x = keras.layers.Conv2DTranspose(16, 3, strides=2, padding="same", activation="relu")(x)
  outputs = keras.layers.Conv2D(1, 1, activation="sigmoid")(x)
  return keras.Model(inputs, outputs, name="unet")

### Entraînement et comparaison

Entraînez ce U-Net comme le premier modèle, avec les mêmes générateurs. Comparez ensuite les deux modèles sur les données de test : accuracy, IoU de la forêt, et prédictions sur les mêmes tuiles. Lequel dessine le mieux les lisières ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
unet = get_pretrained_unet()
unet.summary()
unet.compile(
  # Seul le décodeur, parti de zéro, apprend : un pas d'apprentissage plus grand
  # que pour le premier modèle lui convient
  optimizer=keras.optimizers.Adam(learning_rate=1e-3),
  loss="binary_crossentropy",
  metrics=[
    "accuracy",
    keras.metrics.BinaryIoU(target_class_ids=[1], threshold=0.5, name="iou"),
  ],
)

unet.fit(
  train_generator,
  validation_data=val_generator,
  epochs=epochs,
  callbacks=[
    keras.callbacks.EarlyStopping(patience=5, monitor="val_loss"),
    keras.callbacks.ModelCheckpoint(
      filepath="best_unet.weights.h5",
      monitor="val_accuracy",
      mode="max",
      save_weights_only=True,
      save_best_only=True,
    ),
  ],
)

In [ ]:
# Chargement du meilleur modèle
unet.load_weights("best_unet.weights.h5")

# Scores des deux modèles sur les tuiles de test
results = {
  "U-Net entraîné de zéro": model.evaluate(test_generator, return_dict=True),
  "U-Net pré-entraîné (MobileNetV2)": unet.evaluate(test_generator, return_dict=True),
}
print(pandas.DataFrame(results).T[["accuracy", "iou"]].round(3))

# Les mêmes tuiles de test que pour le premier modèle
preds = unet.predict(X)[..., 0]

for x, y, p in zip(X, Y, preds):
  display_sample(x, y, p)